<a href="https://colab.research.google.com/github/jeffheaton/app_generative_ai/blob/main/t81_559_class_05_5_output_fixing_parsers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# T81-559: Applications of Generative Artificial Intelligence
**Module 5: LangChain: Data Extraction**
* Instructor: [Jeff Heaton](https://sites.wustl.edu/jeffheaton/), McKelvey School of Engineering, [Washington University in St. Louis](https://engineering.wustl.edu/Programs/Pages/default.aspx)
* For more information visit the [class website](https://github.com/jeffheaton/app_generative_ai).

# Module 5 Material

* Part 5.1: The Structured Output Problem [[Video]](https://www.youtube.com/watch?v=62CSR141VRE) [[Notebook]](t81_559_class_05_1_langchain_data.ipynb)
* Part 5.2: Designing Schemas with Pydantic [[Video]](https://www.youtube.com/watch?v=VXm8gPzU3qc) [[Notebook]](t81_559_class_05_2_parsers.ipynb)
* Part 5.3: Validation, Retries, and Refusals [[Video]](https://www.youtube.com/watch?v=dc4fn-W60hg) [[Notebook]](t81_559_class_05_3_pydantic.ipynb)
* Part 5.4: Structured Extraction at Scale [[Video]](https://www.youtube.com/watch?v=jBpkAblQC_U) [[Notebook]](t81_559_class_05_4_custom_parsers.ipynb)
* **Part 5.5: Structured Output Under the Hood** [[Video]](https://www.youtube.com/watch?v=_txWiLjf4bo) [[Notebook]](t81_559_class_05_5_output_fixing_parsers.ipynb)

# Google CoLab Instructions

The following code ensures that Google CoLab is running and maps Google Drive if needed.

In [1]:
import os

try:
    from google.colab import drive, userdata
    COLAB = True
    print("Note: using Google CoLab")
except:
    print("Note: not using Google CoLab")
    COLAB = False

# OpenAI Secrets
if COLAB:
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

# Install needed libraries in CoLab
if COLAB:
    !pip install langchain langchain_openai

Note: using Google CoLab


# 5.5: Structured Output Under the Hood

`with_structured_output` looks like magic: you hand over a class, and prose-generating models start producing typed objects. It is not magic -- it is one of three distinct mechanisms, and knowing which is in use tells you what is actually guaranteed. It also tells you what to do when you must work with a model that supports none of them, such as the small local models of Module 8.

The three mechanisms, from oldest to newest:

1. **Format instructions in the prompt** -- ask nicely, parse, and validate afterward.
2. **Tool calling** -- present the schema as a function the model "calls" with arguments.
3. **Constrained decoding** -- the API masks invalid tokens during generation, making non-conforming output impossible.

In [ ]:
from langchain_openai import ChatOpenAI

MODEL = 'gpt-5.6-luna'

llm = ChatOpenAI(model=MODEL)

from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie recommendation."""
    title: str
    year: int = Field(description="Year of first theatrical release")
    genres: list[str]

## Mechanism 1: Format Instructions in the Prompt

The original approach -- and the one the retired parser classes of earlier course versions were built on -- is to *tell* the model about the schema in the prompt, then parse and validate whatever comes back. LangChain still ships `PydanticOutputParser` for exactly this, and it remains important: it works with **any** model that can follow instructions, including small local ones.

The following prints the format instructions so you can see the man behind the curtain, then runs the full chain.

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate

parser = PydanticOutputParser(pydantic_object=Movie)

print("--- what gets injected into the prompt ---")
print(parser.get_format_instructions())

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the user query.\n{format_instructions}"),
    ("user", "{query}"),
]).partial(format_instructions=parser.get_format_instructions())

chain = prompt | llm | parser
movie = chain.invoke({"query": "Recommend one classic science fiction movie."})
print(movie)

The catch: nothing *forces* the model to comply. A chatty model may wrap the JSON in pleasantries, truncate it, or drift from the schema, and then the parser raises an exception. The retry loop you built in Part 5.3 is the remedy. This fragility is precisely why the industry moved to the next two mechanisms -- but when you run a 4-billion-parameter model on your laptop in Module 8, this is the mechanism you will be using.

## Mechanism 2: Tool Calling

Modern chat models are trained to call functions: given a function signature, they can reply not with prose but with a structured invocation -- a name plus arguments matching the declared parameters. Structured output can ride on this training: present the schema as the one available "function" and require the model to call it. The arguments *are* your object.

In [ ]:
structured_llm = llm.with_structured_output(Movie, method="function_calling")
print(structured_llm.invoke("Recommend one classic science fiction movie."))

This is dramatically more reliable than format instructions because the model was explicitly trained for it -- and it is the exact machinery that agent tools use in Module 7. Its guarantee is still statistical, however: an argument can occasionally be malformed, and validation still runs afterward.

## Mechanism 3: Constrained Decoding

The strongest guarantee comes from OpenAI's native structured outputs (`json_schema` mode). Here the schema is enforced *during generation*: at every step, the API masks away any token that could lead to output violating the schema. Non-conforming output is not repaired or retried -- it is unrepresentable.

In [ ]:
structured_llm = llm.with_structured_output(Movie, method="json_schema")
print(structured_llm.invoke("Recommend one classic science fiction movie."))

Comparing the three:

| Mechanism | How | Shape guarantee | Works with |
|---|---|---|---|
| Format instructions | Schema described in prompt; parse after | None -- parse may fail | Any instruction-following model |
| Tool calling | Schema presented as a callable function | Strong but statistical | Models trained for tool use |
| Constrained decoding | Invalid tokens masked during generation | Absolute (for shape) | APIs with native structured outputs |

When you call `with_structured_output` with no `method` argument, LangChain selects an appropriate mechanism for your model -- which is the right default. Remember from Part 5.3 that even the absolute guarantee is about *shape*, never truth.

## Custom Parsers: The Foundation Layer

All of these conveniences bottom out in the same abstraction: an *output parser* that transforms model text into a Python value and raises `OutputParserException` when it cannot. Subclassing `BaseOutputParser` is rarely necessary anymore, but it remains the escape hatch for formats that are not JSON -- and seeing one demystifies the whole stack.

In [ ]:
from langchain_core.output_parsers import BaseOutputParser
from langchain_core.exceptions import OutputParserException

class CSVLineParser(BaseOutputParser[list[str]]):
    """Parses one comma-separated line into a list of strings."""

    def parse(self, text: str) -> list[str]:
        line = text.strip().splitlines()[0] if text.strip() else ""
        if not line:
            raise OutputParserException("Model returned no content to parse")
        return [item.strip() for item in line.split(",")]

    @property
    def _type(self) -> str:
        return "csv_line_parser"

chain = (
    ChatPromptTemplate.from_template(
        "List exactly {n} {things}. Reply with ONE line of comma-separated values and nothing else."
    )
    | llm
    | CSVLineParser()
)
print(chain.invoke({"n": 5, "things": "programming languages"}))

## Streaming Structured Output

One last practical technique. Structured responses can be large -- imagine a fifty-row extraction -- and users should not stare at a spinner until the last token arrives. If you parse the stream as JSON, you can watch the object *grow*: LangChain's `JsonOutputParser` emits progressively more complete Python structures as tokens arrive. User interfaces use this to render partial results live.

In [ ]:
from langchain_core.output_parsers import JsonOutputParser

json_chain = (
    ChatPromptTemplate.from_template(
        "Recommend 3 classic science fiction movies. "
        "Reply ONLY with a JSON object of the form "
        '{{"movies": [{{"title": ..., "year": ...}}]}}'
    )
    | llm
    | JsonOutputParser()
)

seen = 0
for partial in json_chain.stream({}):
    n = len(partial.get("movies", []))
    if n != seen:
        seen = n
        print(f"...{n} movie(s) so far: {partial['movies'][-1]}")
print("\nfinal:", partial)

## Module Summary

* **Declare, don't parse**: a Pydantic schema plus `with_structured_output` replaces the old parser classes (Part 5.1).
* **The schema is a prompt**: names, descriptions, `Optional`, `Literal`, and nesting are your design tools (Part 5.2).
* **Shape is guaranteed; truth is not**: validators, bounded retries, and defensive wrappers close the gap (Part 5.3).
* **Schema + batch + DataFrame** turns document piles into datasets for pennies (Part 5.4).
* **Three mechanisms** deliver structure -- prompt instructions, tool calling, constrained decoding -- and custom parsers remain the escape hatch for everything else (Part 5.5).

In Module 6, retrieval-augmented generation, the objects you now know how to extract become the metadata that makes document search precise.